In [ ]:
import pandas as pd
drug_nodes = pd.read_pickle("training_data/approved_small_molecule_drugs_review.pkl")
drugs_interactions = pd.read_pickle("training_data/drug_protein_interactions_review.pkl")

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = 'chembl_36/chembl_36_sqlite/chembl_36.db' # or wherever you chembl database is located

print(f"  Size: {Path(DB_PATH).stat().st_size / (1024**3):.1f} GB")

conn = sqlite3.connect(DB_PATH)

  Size: 27.7 GB


In [9]:
# Find which proteins are missing
found_ids = set(proteins_with_sequences['protein_internal_id'].tolist())
missing_proteins = unique_proteins[~unique_proteins['protein_internal_id'].isin(found_ids)]
print(f"Missing {len(missing_proteins)} proteins — attempting fallback lookups...\n")

missing_ids_str = ','.join(map(str, missing_proteins['protein_internal_id'].tolist()))

# ── Fallback 1: SWISS-PROT, any organism ─────────────────────────────────────
fallback1_query = f"""
SELECT DISTINCT
    td.tid as protein_internal_id,
    td.chembl_id as protein_id,
    td.pref_name as protein_name,
    cs.accession as uniprot_id,
    cs.sequence as amino_acid_sequence,
    LENGTH(cs.sequence) as sequence_length,
    td.organism,
    cs.db_source

FROM target_dictionary td
JOIN target_components tc ON td.tid = tc.tid
JOIN component_sequences cs ON tc.component_id = cs.component_id

WHERE td.tid IN ({missing_ids_str})
  AND cs.sequence IS NOT NULL
  AND cs.db_source = 'SWISS-PROT'

ORDER BY td.tid;
"""
fallback1 = pd.read_sql(fallback1_query, conn)
print(f"Fallback 1 (SWISS-PROT, any organism): {len(fallback1)} recovered")

# ── Fallback 2: TREMBL (lower confidence), Homo sapiens ──────────────────────
still_missing = missing_proteins[~missing_proteins['protein_internal_id'].isin(fallback1['protein_internal_id'])]
still_missing_ids_str = ','.join(map(str, still_missing['protein_internal_id'].tolist()))

fallback2_query = f"""
SELECT DISTINCT
    td.tid as protein_internal_id,
    td.chembl_id as protein_id,
    td.pref_name as protein_name,
    cs.accession as uniprot_id,
    cs.sequence as amino_acid_sequence,
    LENGTH(cs.sequence) as sequence_length,
    td.organism,
    cs.db_source

FROM target_dictionary td
JOIN target_components tc ON td.tid = tc.tid
JOIN component_sequences cs ON tc.component_id = cs.component_id

WHERE td.tid IN ({still_missing_ids_str})
  AND cs.sequence IS NOT NULL
  AND td.organism = 'Homo sapiens'

ORDER BY td.tid;
"""
fallback2 = pd.read_sql(fallback2_query, conn)
print(f"Fallback 2 (any db_source, Homo sapiens): {len(fallback2)} recovered")

# ── Fallback 3: Any organism, any source ─────────────────────────────────────
still_missing2 = still_missing[~still_missing['protein_internal_id'].isin(fallback2['protein_internal_id'])]
still_missing2_ids_str = ','.join(map(str, still_missing2['protein_internal_id'].tolist()))

fallback3_query = f"""
SELECT DISTINCT
    td.tid as protein_internal_id,
    td.chembl_id as protein_id,
    td.pref_name as protein_name,
    cs.accession as uniprot_id,
    cs.sequence as amino_acid_sequence,
    LENGTH(cs.sequence) as sequence_length,
    td.organism,
    cs.db_source

FROM target_dictionary td
JOIN target_components tc ON td.tid = tc.tid
JOIN component_sequences cs ON tc.component_id = cs.component_id

WHERE td.tid IN ({still_missing2_ids_str})
  AND cs.sequence IS NOT NULL

ORDER BY td.tid;
"""
fallback3 = pd.read_sql(fallback3_query, conn)
print(f"Fallback 3 (any organism, any source):   {len(fallback3)} recovered")

# ── Combine all results ───────────────────────────────────────────────────────
proteins_with_sequences['source_tier'] = 'primary'
fallback1['source_tier'] = 'fallback_swiss_prot_any_organism'
fallback2['source_tier'] = 'fallback_trembl_human'
fallback3['source_tier'] = 'fallback_any'

all_proteins = pd.concat([proteins_with_sequences, fallback1, fallback2, fallback3], ignore_index=True)

# Keep first (best) match per protein
all_proteins = all_proteins.drop_duplicates(subset='protein_internal_id', keep='first')

# Report truly unresolvable proteins
truly_missing = unique_proteins[~unique_proteins['protein_internal_id'].isin(all_proteins['protein_internal_id'])]
print(f"\n── Summary ──────────────────────────────────────────")
print(f"  Total unique proteins:     {len(unique_proteins)}")
print(f"  Sequences found:           {len(all_proteins)}")
print(f"  Still missing (no seq):    {len(truly_missing)}")

if len(truly_missing) > 0:
    print("\nProteins with NO sequence in ChEMBL at all:")
    print(truly_missing[['protein_internal_id', 'protein_id', 'protein_name']].to_string(index=False))

all_proteins.to_csv("proteins_for_embedding_extended.csv", index=False)
print("\n✓ Saved to proteins_for_embedding_extended.csv")

Missing 836 proteins — attempting fallback lookups...

Fallback 1 (SWISS-PROT, any organism): 662 recovered
Fallback 2 (any db_source, Homo sapiens): 0 recovered
Fallback 3 (any organism, any source):   176 recovered

── Summary ──────────────────────────────────────────
  Total unique proteins:     2053
  Sequences found:           2053
  Still missing (no seq):    0

✓ Saved to proteins_for_embedding_extended.csv


In [ ]:
all_proteins

,protein_internal_id,protein_id,protein_name,uniprot_id,amino_acid_sequence,sequence_length,source_tier,organism,db_source
0,1,CHEMBL2074,Maltase-glucoamylase,O43451,MARKKLKKFTTLEIVLSVLLLVLFIISIVLIVLLAKESLKSTAPDP...,2753,primary,NaN,NaN
1,2,CHEMBL1971,ATP-binding cassette sub-family C member 9,O60706,MSLSFCGNNISSYNINDGVLQNSCFVDALNLVPHVFLLFITFPILF...,1549,primary,NaN,NaN
2,3,CHEMBL1827,"cGMP-specific 3',5'-cyclic phosphodiesterase",O76074,MERAGPSFGQQRQQQQPQQQKQQQRDQDSVEAWLDDHWDFTFSYFV...,875,primary,NaN,NaN
3,4,CHEMBL1859,Voltage-dependent T-type calcium channel subun...,O95180,MTEGARAADEVRVPLGAPPPGPAALVGASPESPGAPGREAERGSEL...,2353,primary,NaN,NaN
4,6,CHEMBL202,Dihydrofolate reductase,P00374,MVGSLNCIVAVSQNMGIGKNGDLPWPPLRNEFRYFQRMTTTSSVEG...,187,primary,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2050,119669,CHEMBL4523952,NS5,A0A143MHK7,EFGKAKGSRAIWYMWLGARFLEFEALGFLNEDHWMGRENSGGGVEG...,340,fallback_any,Zika virus,TREMBL
2051,119671,CHEMBL4523954,Nonstructural protein 3,A0A024AXB9,AETDEDHAHWLEARMLLDNIYLQDGLIASLYRPEADKVAAIEGEFK...,66,fallback_any,Zika virus,TREMBL
2052,120176,CHEMBL4739840,Histone-lysine N-methyltransferase EZH2,B5DFE2,MGQTGKKSEKGPVCWRKRVKSEYMRLRQLKRFRRADEVKTMFSSNR...,746,fallback_any,Rattus norvegicus,TREMBL
2053,122055,CHEMBL6066151,Methionine--tRNA ligase,A0A3N3ZFK4,MAEKETFYITTPIYYPSGKLHIGNSYTTIACDAIARYKRLMGFDVF...,669,fallback_any,Enterococcus faecalis,TREMBL


In [12]:
protein_nodes = all_proteins
import torch
from transformers import AutoTokenizer, EsmModel 

/home/joe/projects/pharmacology-graph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
import numpy as np
from tqdm import tqdm
import gc

print("\n" + "="*80)
print("esm2_t48_15B_UR50D")
print("="*80)

# Configuration
ESM_MODEL = "facebook/esm2_t36_3B_UR50D"  # 3B params, 1280-dim
MAX_LENGTH = 1024  # ESM-2 max sequence length
BATCH_SIZE = 4  # Process 4 proteins at once
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nDevice: {DEVICE}")
print(f"Model: {ESM_MODEL}")
print(f"Batch size: {BATCH_SIZE}")

# Load model
print("\nLoading ESM-2 model...")
tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL)
model = EsmModel.from_pretrained(ESM_MODEL)
model.to(DEVICE)
model.eval()
print("✓ Model loaded")

# Function to compute embeddings in batches
def compute_esm2_embeddings_batch(sequences, batch_size=4):
    """Compute ESM-2 embeddings for multiple sequences."""
    
    all_embeddings = []
    num_batches = (len(sequences) + batch_size - 1) // batch_size
    
    print(f"\nComputing embeddings in {num_batches} batches...")
    
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch = sequences[i:i+batch_size]
        
        # Tokenize batch
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Mean pooling over sequence length
        embeddings = outputs.last_hidden_state.mean(dim=1)  # (batch_size, 1280)
        embeddings = embeddings.cpu().numpy()
        
        all_embeddings.append(embeddings)
        
        # Clear GPU memory
        del inputs, outputs, embeddings
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    
    return np.vstack(all_embeddings)

# Compute embeddings
sequences = protein_nodes['amino_acid_sequence'].tolist()

print(f"\nProcessing {len(sequences)} protein sequences...")
print(f"Estimated time: {len(sequences) * 2 / 60:.1f} minutes (CPU) or {len(sequences) * 0.5 / 60:.1f} minutes (GPU)")

embeddings = compute_esm2_embeddings_batch(sequences, batch_size=BATCH_SIZE)

print(f"\n✓ Computed {len(embeddings)} embeddings")
print(f"  Embedding shape: {embeddings.shape}")
print(f"  Memory usage: {embeddings.nbytes / (1024**2):.1f} MB")



esm2_t48_15B_UR50D

Device: cuda
Model: facebook/esm2_t36_3B_UR50D
Batch size: 4

Loading ESM-2 model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  8.77it/s]
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t36_3B_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded

Processing 2053 protein sequences...
Estimated time: 68.4 minutes (CPU) or 17.1 minutes (GPU)

Computing embeddings in 514 batches...


100%|██████████| 514/514 [10:43<00:00,  1.25s/it]


✓ Computed 2053 embeddings
  Embedding shape: (2053, 2560)
  Memory usage: 20.0 MB


In [14]:
proteins_with_sequences = protein_nodes.copy()
# Add embeddings to dataframe
proteins_with_sequences['esm2_embedding'] = list(embeddings)

# Save
print("\nSaving results...")
proteins_with_sequences.to_pickle('protein_nodes_with_embeddings_extended.pkl', protocol=4)
print("✓ Saved to protein_nodes_with_embeddings_extended.pkl")


Saving results...
✓ Saved to protein_nodes_with_embeddings_extended.pkl
